# Regression with ARIMA Errors

Standard linear regression assumes that the errors $\varepsilon_t$ are
independent and identically distributed (i.i.d.).  When we apply regression
to **time series** data, this assumption almost always fails — the errors
are correlated over time.

**Dynamic regression** (FPP3 Chapter 10) fixes this by allowing the error
term to follow an ARIMA process:

$$
y_t = \beta_0 + \beta_1 x_{1,t} + \cdots + \beta_k x_{k,t} + \eta_t
$$

where $\eta_t$ is an ARIMA(p,d,q) process instead of white noise.

This is exactly what the **"X" in SARIMAX** provides — the ability to add
e**X**ogenous regressors to a SARIMA model.

This notebook covers:
1. Why regression errors are autocorrelated for time series
2. How regression with ARIMA errors works
3. The SARIMAX API for dynamic regression
4. A practical example with macroeconomic data
5. Key rules for working with exogenous variables

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. The Problem: Autocorrelated Regression Errors

Consider a simple regression: predict **real personal consumption** from
**real GDP** using quarterly US macroeconomic data.

If we fit an OLS regression, the residuals will almost certainly be
autocorrelated — violating the i.i.d. assumption.

In [ ]:
# Load macroeconomic data
macro = pd.read_csv(DATA_DIR / "macrodata.csv", index_col=0, parse_dates=True)
macro.index.freq = "QS-OCT"

# Select key columns
macro = macro[["realgdp", "realcons", "cpi", "unemp"]].copy()
macro.columns = ["RealGDP", "RealCons", "CPI", "Unemployment"]

print(f"Shape: {macro.shape}")
print(f"Date range: {macro.index[0].date()} to {macro.index[-1].date()}")
macro.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(macro["RealGDP"], label="Real GDP")
axes[0].plot(macro["RealCons"], label="Real Consumption")
axes[0].set_title("Real GDP and Real Consumption")
axes[0].set_ylabel("Billions of Dollars")
axes[0].legend()

axes[1].scatter(macro["RealGDP"], macro["RealCons"], alpha=0.5, s=15)
axes[1].set_xlabel("Real GDP")
axes[1].set_ylabel("Real Consumption")
axes[1].set_title("Consumption vs GDP (scatter)")

plt.tight_layout()
plt.show()

print("Strong linear relationship — but the observations are ordered in time,")
print("so the errors from a simple regression will be autocorrelated.")

In [ ]:
# Fit OLS regression: RealCons ~ RealGDP
ols = LinearRegression()
X_ols = macro[["RealGDP"]].values
y_ols = macro["RealCons"].values
ols.fit(X_ols, y_ols)

ols_resid = y_ols - ols.predict(X_ols)

print(f"OLS: RealCons = {ols.intercept_:.2f} + {ols.coef_[0]:.4f} * RealGDP")
print(f"R² = {ols.score(X_ols, y_ols):.4f}")

In [ ]:
# Plot residuals and their ACF
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(macro.index, ols_resid)
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[0].set_title("OLS Residuals Over Time")
axes[0].set_ylabel("Residual")

plot_acf(ols_resid, lags=20, ax=axes[1], title="ACF of OLS Residuals")
plot_pacf(ols_resid, lags=20, ax=axes[2], title="PACF of OLS Residuals", method="ywm")

plt.tight_layout()
plt.show()

print("The residuals are HIGHLY autocorrelated — they drift slowly,")
print("showing that consecutive errors are positively correlated.")
print("OLS standard errors and confidence intervals are invalid here.")

In [ ]:
# Ljung-Box test confirms autocorrelation
lb = acorr_ljungbox(ols_resid, lags=[4, 8, 12], return_df=True)
print("Ljung-Box test on OLS residuals:")
print(lb)
print()
print("All p-values ≈ 0 → residuals are NOT white noise.")
print("Standard regression is inadequate for this time series.")

---
## 2. The Solution: Regression with ARIMA Errors

Instead of assuming $\varepsilon_t \sim \text{i.i.d.}$, we let the errors
follow an ARIMA process:

$$
y_t = \beta_0 + \beta_1 x_{1,t} + \cdots + \beta_k x_{k,t} + \eta_t
$$

$$
\phi(B)(1-B)^d \eta_t = \theta(B) \varepsilon_t, \quad \varepsilon_t \sim \text{WN}(0, \sigma^2)
$$

where:
- $\beta_i$ are the regression coefficients (effect of each predictor)
- $\eta_t$ is the error term, which follows an ARIMA(p,d,q) process
- $\phi(B)$ and $\theta(B)$ are the AR and MA polynomials
- $\varepsilon_t$ is the final white noise innovation

**Key distinction from plain ARIMA:** we are not just modelling $y_t$ with
its own past — we are also using external predictors $x_{1,t}, \ldots, x_{k,t}$
to explain variation that the ARIMA structure alone would miss.

**Key distinction from plain regression:** we properly account for the
autocorrelation in the errors, giving us valid inference and better forecasts.

---
## 3. The SARIMAX API

In statsmodels, the `SARIMAX` class handles both SARIMA and dynamic regression:

```python
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    endog=y,              # the target series
    exog=X,               # exogenous predictors (DataFrame or array)
    order=(p, d, q),      # ARIMA order for the errors
    seasonal_order=(P, D, Q, m),  # optional seasonal ARIMA
)
result = model.fit(disp=False)

# IMPORTANT: you must provide future exog values for forecasting!
forecast = result.get_forecast(steps=h, exog=future_X)
```

**Critical rule:** you must supply `exog` for **both fitting and forecasting**.
The forecast exog must have the same columns and exactly `steps` rows.

---
## 4. Example: Forecasting Consumption from GDP

We will forecast **Real Personal Consumption** using **Real GDP** as an
exogenous predictor, with ARIMA errors to capture the time-series structure.

In [ ]:
# Train/test split — hold out last 20 quarters (5 years)
HORIZON = 20
train = macro.iloc[:-HORIZON]
test = macro.iloc[-HORIZON:]

print(f"Train: {len(train)} quarters ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} quarters ({test.index[0].date()} to {test.index[-1].date()})")

fig, ax = plt.subplots()
ax.plot(train["RealCons"], label="Train")
ax.plot(test["RealCons"], label="Test", color="tab:orange")
ax.axvline(test.index[0], color="grey", linestyle="--", alpha=0.7)
ax.set_title("Train / Test Split — Real Consumption")
ax.set_ylabel("Billions of Dollars")
ax.legend()
plt.tight_layout()
plt.show()

### 4a. Baseline: Pure ARIMA (no exogenous variables)

First, let us fit a plain ARIMA on consumption — no GDP predictor.

In [ ]:
# Pure ARIMA baseline — no exogenous
arima_model = SARIMAX(
    train["RealCons"],
    order=(1, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
arima_result = arima_model.fit(disp=False)
print(arima_result.summary().tables[0])
print(f"\nAIC: {arima_result.aic:.2f}")

In [ ]:
# Forecast with pure ARIMA
arima_fc = arima_result.get_forecast(steps=HORIZON)
arima_pred = arima_fc.predicted_mean

arima_mae = mean_absolute_error(test["RealCons"], arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test["RealCons"], arima_pred))

print(f"Pure ARIMA(1,1,1) — no exogenous:")
print(f"  MAE:  {arima_mae:.2f}")
print(f"  RMSE: {arima_rmse:.2f}")

### 4b. Dynamic Regression: ARIMA with GDP as Exogenous

Now let us add Real GDP as an exogenous predictor.  The SARIMAX class
estimates the regression coefficients **and** the ARIMA error parameters
simultaneously via maximum likelihood.

In [ ]:
# Dynamic regression: RealCons ~ RealGDP + ARIMA(1,1,1) errors
dynreg_model = SARIMAX(
    endog=train["RealCons"],
    exog=train[["RealGDP"]],
    order=(1, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
dynreg_result = dynreg_model.fit(disp=False)
print(dynreg_result.summary())

In [ ]:
print("Key things to notice in the summary:")
print("  1. The 'RealGDP' coefficient — the marginal effect of GDP on consumption")
print("  2. The ar.L1 and ma.L1 coefficients — the ARIMA structure of the errors")
print("  3. AIC is reported — compare with the pure ARIMA AIC")
print()
print(f"Pure ARIMA AIC:       {arima_result.aic:.2f}")
print(f"Dynamic Regression AIC: {dynreg_result.aic:.2f}")
print()
if dynreg_result.aic < arima_result.aic:
    print("Lower AIC → adding GDP improves the model.")
else:
    print("AIC did not improve — GDP may already be captured by the ARIMA structure.")

In [ ]:
# Forecast — MUST provide exog for the forecast period
# In practice, you need future GDP values.  Here we use the actual test values
# to demonstrate the API (in a real scenario, you would use GDP forecasts).
dynreg_fc = dynreg_result.get_forecast(
    steps=HORIZON,
    exog=test[["RealGDP"]],
)
dynreg_pred = dynreg_fc.predicted_mean
dynreg_ci = dynreg_fc.conf_int(alpha=0.05)

dynreg_mae = mean_absolute_error(test["RealCons"], dynreg_pred)
dynreg_rmse = np.sqrt(mean_squared_error(test["RealCons"], dynreg_pred))

print(f"Dynamic Regression ARIMA(1,1,1) + GDP:")
print(f"  MAE:  {dynreg_mae:.2f}")
print(f"  RMSE: {dynreg_rmse:.2f}")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["RealCons"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(arima_pred, label="Pure ARIMA", color="tab:red", linestyle="--")
ax.plot(dynreg_pred, label="Dynamic Reg (ARIMA + GDP)", color="tab:green", linestyle="--")
ax.fill_between(
    dynreg_ci.index, dynreg_ci.iloc[:, 0], dynreg_ci.iloc[:, 1],
    color="tab:green", alpha=0.1, label="95% CI (dyn reg)",
)
ax.set_title("Pure ARIMA vs Dynamic Regression")
ax.set_ylabel("Real Consumption")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Residual Diagnostics

A well-specified dynamic regression model should have white noise residuals.

In [ ]:
fig = dynreg_result.plot_diagnostics(figsize=(14, 10))
plt.suptitle("Dynamic Regression — Residual Diagnostics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare residuals: OLS vs dynamic regression
lb_dynreg = acorr_ljungbox(dynreg_result.resid.dropna(), lags=[4, 8, 12], return_df=True)
print("Ljung-Box test on dynamic regression residuals:")
print(lb_dynreg)
print()
print("Compare with OLS residuals (all p ≈ 0).")
print("The ARIMA error structure should produce much larger p-values,")
print("indicating the residuals are closer to white noise.")

---
## 6. Multiple Exogenous Predictors

We can add more than one exogenous variable.  Let us try using both
GDP and Unemployment to predict consumption.

In [ ]:
# Multiple exogenous: GDP + Unemployment
exog_cols = ["RealGDP", "Unemployment"]

multi_model = SARIMAX(
    endog=train["RealCons"],
    exog=train[exog_cols],
    order=(1, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
multi_result = multi_model.fit(disp=False)

# Forecast
multi_fc = multi_result.get_forecast(steps=HORIZON, exog=test[exog_cols])
multi_pred = multi_fc.predicted_mean

multi_mae = mean_absolute_error(test["RealCons"], multi_pred)
multi_rmse = np.sqrt(mean_squared_error(test["RealCons"], multi_pred))

print(f"Dynamic Regression ARIMA(1,1,1) + GDP + Unemployment:")
print(f"  MAE:  {multi_mae:.2f}")
print(f"  RMSE: {multi_rmse:.2f}")
print(f"  AIC:  {multi_result.aic:.2f}")

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    "Model": [
        "Pure ARIMA(1,1,1)",
        "ARIMA(1,1,1) + GDP",
        "ARIMA(1,1,1) + GDP + Unemp",
    ],
    "AIC": [arima_result.aic, dynreg_result.aic, multi_result.aic],
    "MAE": [arima_mae, dynreg_mae, multi_mae],
    "RMSE": [arima_rmse, dynreg_rmse, multi_rmse],
}).round(2)

print(comparison.to_string(index=False))

---
## 7. Common Pitfalls

### Pitfall 1: Forgetting exog during forecasting

In [ ]:
# This will raise an error — exog was used in fitting but not forecasting
try:
    bad_fc = dynreg_result.get_forecast(steps=HORIZON)
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
    print()
    print("LESSON: if you fit with exog, you MUST provide exog for forecasting.")
    print("The exog array must have exactly 'steps' rows and the same columns.")

### Pitfall 2: Using the target's own lagged values as exogenous

You should **not** use lagged values of $y_t$ as exogenous variables —
that is what the ARIMA part already does.  Exogenous variables must be
**external** to the series being forecast.

### Pitfall 3: Exogenous values must be known for the forecast period

If your exogenous variable is not known in advance (e.g., future GDP),
you need a separate forecast of that variable first, which adds uncertainty.
The best exogenous variables are **deterministic** — things like:
- Calendar indicators (day of week, month, holiday)
- Fourier terms for seasonality
- Known future events (scheduled promotions, policy changes)
- Trend counters (time index)

---
## 8. How Dynamic Regression Differs from Related Approaches

| Approach | Errors | Predictors | Implemented as |
|----------|--------|------------|----------------|
| OLS Regression | i.i.d. $\varepsilon_t$ | Yes | `LinearRegression` |
| Pure ARIMA | ARIMA $\eta_t$ | No | `SARIMAX(endog, order=...)` |
| **Dynamic Regression** | **ARIMA $\eta_t$** | **Yes** | **`SARIMAX(endog, exog=X, order=...)`** |
| VAR | Vector AR | Multiple endogenous | `VAR` (Chapter 12) |

Dynamic regression is the natural combination: it uses external predictors
(like regression) while properly modelling the autocorrelated error structure
(like ARIMA).

---
## Summary

- Standard regression on time series data produces **autocorrelated residuals**,
  violating the i.i.d. assumption and invalidating standard errors.
- **Dynamic regression** (regression with ARIMA errors) solves this by letting
  the error term $\eta_t$ follow an ARIMA process.
- In statsmodels, use `SARIMAX(endog=y, exog=X, order=(p,d,q))` — the same
  class used for pure SARIMA, but with the `exog` argument.
- **You must supply `exog`** for both fitting and forecasting.  The forecast
  exog must have exactly `steps` rows.
- Exogenous variables should ideally be **known in advance** for the forecast
  period (holidays, calendar effects, Fourier terms).
- The model estimates regression coefficients and ARIMA parameters jointly
  via maximum likelihood — no two-step procedure needed.
- Always check residual diagnostics to verify the model is adequate.

In [ ]:
print("Next notebook: SARIMAX with exogenous variables — the restaurant")
print("visitors example with holiday effects from Portilla Section 06-07.")